# Diffusion Basics

Companion notebook for the diffusion basics learning session. You will visualize the forward noising process and train a tiny epsilon-prediction model.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
torch.manual_seed(0)

In [ ]:
steps = 100
betas = torch.linspace(1e-4, 0.02, steps, device=device)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, noise):
    a = alpha_bar[t].view(-1, 1)
    return a.sqrt() * x0 + (1.0 - a).sqrt() * noise

In [ ]:
# Clean toy trajectory flattened into D=40.
T = 20
time = torch.linspace(0, 1, T, device=device)
traj = torch.stack([5 * time, 2 * torch.sin(3.14 * time)], dim=-1).reshape(1, -1)

plt.figure(figsize=(12, 3))
for i, tval in enumerate([0, 20, 50, 99]):
    noise = torch.randn_like(traj)
    xt = q_sample(traj, torch.tensor([tval], device=device), noise).reshape(T, 2).detach().cpu()
    plt.subplot(1, 4, i + 1)
    plt.plot(xt[:, 0], xt[:, 1], marker='o')
    plt.title(f't={tval}')
    plt.axis('equal')
plt.show()

In [ ]:
class TinyDenoiser(nn.Module):
    def __init__(self, dim, steps=100, hidden=128):
        super().__init__()
        self.time = nn.Embedding(steps, hidden)
        self.net = nn.Sequential(nn.Linear(dim + hidden, hidden), nn.ReLU(), nn.Linear(hidden, dim))
    def forward(self, xt, t):
        return self.net(torch.cat([xt, self.time(t)], dim=-1))

model = TinyDenoiser(T * 2, steps).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train on a small family of sine trajectories.
def sample_clean(batch):
    amp = torch.rand(batch, 1, device=device) * 3
    phase = torch.rand(batch, 1, device=device) * 3.14
    x = 5 * time[None, :].expand(batch, T)
    y = amp * torch.sin(3.14 * time[None, :] + phase)
    return torch.stack([x, y], dim=-1).reshape(batch, -1)

for step in range(1000):
    x0 = sample_clean(128)
    t = torch.randint(0, steps, (x0.size(0),), device=device)
    noise = torch.randn_like(x0)
    xt = q_sample(x0, t, noise)
    loss = F.mse_loss(model(xt, t), noise)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 250 == 0:
        print(step, float(loss))

Interview drill: explain why the target is noise and how to recover an estimate of the clean sample from predicted epsilon.